<a href="https://colab.research.google.com/github/nishantchangz-commits/nick/blob/main/day8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -U langchain-community langchain-huggingface faiss-cpu python-dotenv langchain-openai

import os
from dotenv import load_dotenv
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

load_dotenv()

def load_documents(path: str):
  loader = TextLoader(path, encoding="utf-8")
  documents = loader.load()
  print(f"loaded {len(documents)} document(s) from {path}")
  return documents

def split_documents(documents, chunk_size: int = 400, chunk_overlap: int = 60):
  splitter = RecursiveCharacterTextSplitter(
      chunk_size=chunk_size,
      chunk_overlap=chunk_overlap,
      separators=["\n\n", "\n", ".", "", ""]
  )
  chunks = splitter.split_documents(documents)
  print(f"split into {len(chunks)} chunks")
  return chunks

def build_vectorstore(chunks):
  embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
  vectorstore = FAISS.from_documents(chunks, embeddings)
  print("vector store built")
  return vectorstore

def build_retriever(vectorstore, k: int = 3):
  return vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": k})

def build_llm():
  from langchain_openai import ChatOpenAI
  if not os.getenv("OPENAI_API_KEY"):
    raise EnvironmentError("OPENAI_API_KEY not found. Either set it in a .env file, or "
                           "switch build_llm to use chatollama for a free local model "
                           "(see the docstring above)")
  return ChatOpenAI(model="gpt-4o-mini", temperature=0)

def build_rag_chain(retriever, llm):
  # You need to provide a template string for ChatPromptTemplate.from_template()
  prompt = ChatPromptTemplate.from_template("Answer the question based only on the following context:\n{context}\nQuestion: {question}")

  def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

  chain = (
      {"context": retriever | format_docs, "question": RunnablePassthrough()}
      | prompt
      | llm
      | StrOutputParser()
  )
  return chain

if __name__ == "__main__":
  doc_path = "sample_docs/company_handbook.txt"

  # Create the directory and an empty file if they don't exist
  os.makedirs(os.path.dirname(doc_path), exist_ok=True)
  if not os.path.exists(doc_path) or os.stat(doc_path).st_size == 0:
      with open(doc_path, 'w') as f:
          f.write("This is a sample company handbook. Full-time employees receive 15 vacation days per year. Remote work is allowed for up to 3 days per week, with manager approval. If company equipment is not returned on time after separation, the employee will be charged for its full replacement cost.") # Add some sample content
      print(f"Created or populated {doc_path} with sample content.")

  documents = load_documents(doc_path)
  chunks = split_documents(documents)
  vectorstore = build_vectorstore(chunks)
  retriever = build_retriever(vectorstore)
  llm = build_llm()
  rag_chain = build_rag_chain(retriever, llm)
  questions = [
      "how many vacation days do full-time employee get?",
      "can i work remotely, and how many days per week?",
      "what happen if i dont return my equipment on time?"

  ]
  for q in questions:
    print(f"\nQ: {q}")
    answer = rag_chain.invoke(q)
    print(f"A: {answer}")

Created or populated sample_docs/company_handbook.txt with sample content.
loaded 1 document(s) from sample_docs/company_handbook.txt
split into 1 chunks


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

vector store built


OSError: OPENAI_API_KEY not found. Either set it in a .env file, or switch build_llm to use chatollama for a free local model (see the docstring above)

In [ ]:
# To get started, make sure you have an OpenAI API key.
# You can add it to Colab's Secrets manager by clicking on the '🔑' icon in the left sidebar.
# Then, add a new secret named 'OPENAI_API_KEY' and paste your key there.
# Remember to enable 'Access notebook secrets' for this notebook.
import os
from google.colab import userdata

# Set the API key as an environment variable
os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')

print("OPENAI_API_KEY has been set.")

SecretNotFoundError: Secret OPENAI_API_KEY does not exist.